# Урок 18 (продължение): Касови бележки, които доказват, че *човек* е упълномощил действието

Урокът доказва какво е направил **агентът** и какво е решил **порталът**. Този тефтер добавя липсващата част: доказателство, че **именуван човек** е одобрил **точното** действие — отделен подпис, държан от човек, върху пълното канонично действие, проверен офлайн.

И двете произведения тук използват **същата форма на плик, както касовите бележки от урока**: плосък полезен товар с поле `type`, подписан с Ed25519 върху SHA-256 на каноничните байтове на полезния товар, с прикачен структуриран обект `signature` (и изключен от подписаните байтове). Касовата бележка за одобрение е нов `type` (`human.approval.v1`) заедно с типа действие, така че един `verify_chain` обхваща и двата вида произведения със същия кодов път, който изградихте в главния тефтер. Това също е формата на пътя за съ-сигнатура в Internet-Draft, който урокът следва (draft-farley-acta-signed-receipts).

Едно умишлено подобрение спрямо демонстратора в главния тефтер: тук проверяващият разрешава `signature.key_id` спрямо **фиксиран регистър на ключове**, вместо да се доверява на публичен ключ, носен вътре в касовата бележка. Това е производствената позиция, която препоръчва самият списък с проверки на урока („публикувайте публичния ключ за проверка“), и именно той превръща фалшифицирането във отказ вместо заобикаляне с предоставен от потребителя ключ.

Правилото, което този тефтер преподава: **подписаното одобрение само по себе си не е правомощие.** Правомощие съществува само ако касовата бележка за одобрение и касовата бележка за действие все още се отнасят към същото канонично действие по време на изпълнение, под версия на политика, ключ и срок на валидност, които все още са валидни, и одобрение, което още не е било използвано. Всеки отказ е с **отделна причина**, така че можете да разграничите *правомощията да са изтекли* от *изпълненото действие да се е променило*.


In [1]:
# These are already the Lesson 18 dependencies — no new packages.
# %pip install pynacl jcs
import base64, copy, hashlib
from jcs import canonicalize                      # RFC 8785 canonical JSON
from nacl.signing import SigningKey, VerifyKey
# CryptoError is the common base of BadSignatureError AND the ValueError pynacl
# raises for a wrong-length signature — catch the base so verification fails
# closed on ANY bad signature, not just the forged-but-correct-length one.
from nacl.exceptions import CryptoError

# Same helpers as the main notebook.
def b64url_nopad(data: bytes) -> str:
    return base64.urlsafe_b64encode(data).decode("ascii").rstrip("=")

def b64url_decode(s: str) -> bytes:
    return base64.urlsafe_b64decode(s + "=" * ((4 - len(s) % 4) % 4))

def sha256_canonical(obj) -> str:
    """SHA-256 of an object's JCS-canonical JSON form (same helper as the lesson)."""
    return f"sha256:{hashlib.sha256(canonicalize(obj)).hexdigest()}"

## Точното действие

Единицата за одобрение е **каноничният обект на действие** — не някакъв неясен етикет като "одобряване на възстановяване", а точното, изцяло специфицирано действие. Подписването на целия обект (и извеждането на дигест от него) ни позволява по-късно да докажем, че човекът е одобрил *това* и нищо друго.


In [2]:
action = {
    "action_type": "refund.issue",
    "params": {"order_id": "A-1029", "amount_usd": 4200, "to": "acct_88"},
    "policy_id": "refunds-v3",
}
print("action digest:", sha256_canonical(action))

action digest: sha256:fba342ad8447b491a089d7a09d4ac58f1a835c504e58f8d832db04f65bb62a25


## Един плик, два органа

Всеки разписък е пликът на урока: плосък полезен товар с поле `type`, плюс обект `signature` (`alg`, `sig`, `key_id`), който **не** е част от подписаните байтове. `verify_envelope` е общата структурна + проверка на подписа за двата вида разписки; коя **регистрирана ключова регистрация** използва, за да резолвира `signature.key_id` е това, което държи органите разделени:

- **разписка за одобрение** (`human.approval.v1`) — наименуван одобряващ, пълно канонично действие **и неговото хеширане (digest)**, `policy_version`, времеви отметки за издаване и изтичане. Еднократната консумация се следи на ниво верига.
- **разписка за действие** (`agent.action.v1`) — идентичност на агента, `run_id`, същото канонично хеширане на действие, резултат от изпълнението + времева отметка и `parent_approval_ref`: `receipt_hash` на одобрението, същата конвенция като `previous_receipt_hash` във веригата на урока.

Общото поле `action_digest` е връзката, от която зависи сливането. `key_id` се намира в обекта на подписа само като подсказка за търсене: пренасочването му към друг регистриран ключ прави проверката на подписа неуспешна, така че то не дава нищо.


In [3]:
# ---- pinned key registries: SEPARATE authorities, one envelope shape ----------
# Published out of band (the lesson checklist's JWK-Set pattern); the verifier
# NEVER trusts a key carried inside a receipt.
approver_sk = SigningKey.generate()
agent_sk    = SigningKey.generate()
APPROVER_KEYS = {"approver-key-1": b64url_nopad(bytes(approver_sk.verify_key))}
AGENT_KEYS    = {"agent-key-1":    b64url_nopad(bytes(agent_sk.verify_key))}

# The policy the approval is granted under. If this moves after approval, the
# approval is STALE even though its signature still verifies.
CURRENT_POLICY = {"policy_version": "refunds-v3"}

def sign_receipt(payload: dict, sk: SigningKey, key_id: str) -> dict:
    """Same signing pipeline as the lesson: Ed25519 over the SHA-256 of the
    canonical payload; the signature object is NOT part of the signed bytes."""
    message_hash = hashlib.sha256(canonicalize(payload)).digest()
    return {
        **payload,
        "signature": {"alg": "EdDSA", "sig": b64url_nopad(sk.sign(message_hash).signature), "key_id": key_id},
    }

def verify_envelope(receipt, expected_type: str, trusted_keys: dict):
    """The SHARED verifier contract for any receipt kind; the caller picks which
    pinned registry (authority) resolves key_id. Fails closed on ANY
    attacker-shaped input: malformed is a refusal, never a crash."""
    if not isinstance(receipt, dict) or not isinstance(receipt.get("signature"), dict):
        return (False, "receipt malformed (not an object with a signature object)")
    sig_obj = receipt["signature"]
    if sig_obj.get("alg") != "EdDSA":
        return (False, "unsupported signature alg")
    if receipt.get("type") != expected_type:
        return (False, f"wrong receipt type (expected {expected_type})")
    # Key freshness is part of authority: a key_id rotated out of the pinned
    # registry confers nothing, even with a valid signature.
    pub = trusted_keys.get(sig_obj.get("key_id"))
    if pub is None:
        return (False, f"stale authority: key_id {sig_obj.get('key_id')!r} is not in the pinned registry (unknown or rotated out)")
    # Reconstruct the signed bytes exactly as the lesson does: everything except
    # the signature object, canonicalized, hashed.
    payload = {k: v for k, v in receipt.items() if k != "signature"}
    try:
        message_hash = hashlib.sha256(canonicalize(payload)).digest()
        VerifyKey(b64url_decode(pub)).verify(message_hash, b64url_decode(sig_obj.get("sig") or ""))
    except (CryptoError, TypeError, ValueError, base64.binascii.Error):
        return (False, "signature invalid (forged, tampered, or malformed)")
    return (True, "envelope ok")

def human_approval(action, approver_id, approved_at, sk=approver_sk,
                   key_id="approver-key-1", policy_version=None, expires_at=None):
    # deepcopy: the receipt must be an immutable record of what was approved —
    # a live reference would let a later mutation of `action` silently change the
    # signed payload. Digest the SNAPSHOT so the two can never diverge.
    approved_action = copy.deepcopy(action)
    payload = {
        "type": "human.approval.v1",
        "approver_id": approver_id,
        "action": approved_action,                       # the FULL canonical action
        "action_digest": sha256_canonical(approved_action),  # the join field
        "policy_version": policy_version or CURRENT_POLICY["policy_version"],
        "approved_at": approved_at,                      # ISO-8601 Zulu, like the lesson
        "expires_at": expires_at or approved_at[:11] + "23:59:59Z",
    }
    return sign_receipt(payload, sk, key_id)

In [4]:
approval = human_approval(action, "alice@ops (WebAuthn)", "2026-07-08T15:04:05Z",
                          expires_at="2026-07-08T15:19:05Z")
print(verify_envelope(approval, "human.approval.v1", APPROVER_KEYS))
print("binds digest:", approval["action_digest"][:23], "…  under", approval["policy_version"])

(True, 'envelope ok')
binds digest: sha256:fba342ad8447b491 …  under refunds-v3


## `verify_chain`: където всъщност се решава свързването

`verify_chain` **не е** обвивка за удобство над две проверки на подпис. Това е мястото, където се проверяват заедно споделеният каноничен `action_digest`, политиката/ключът/свежестта на валидността на одобрението и **еднократната консумация** на одобрението, спрямо действието, което се изпълнява *в този момент*.

Всяка грешка отказва с **отличителна причина**, така че читателят на отказа може да разбере дали властта е остаряла (политиката се е преместила, ключът е ротиран, одобрението е изтекло, одобрението е консумирано) или изпълненото действие се е променило под все още валидно одобрение (замяна на дайджест).


In [5]:
def receipt_hash(receipt: dict) -> str:
    """Content-derived id of a COMPLETE receipt (including its signature) —
    the same convention as previous_receipt_hash in the lesson's chain."""
    return sha256_canonical(receipt)

def agent_receipt(action, approval, executed_at, sk=agent_sk, key_id="agent-key-1"):
    executed_action = copy.deepcopy(action)    # snapshot, same reason as the approval
    payload = {
        "type": "agent.action.v1",
        "agent_id": "agent:refunds-bot",
        "run_id": "run-0001",
        "action": executed_action,
        "action_digest": sha256_canonical(executed_action),  # same join field
        "parent_approval_ref": receipt_hash(approval),
        "outcome": "performed",
        "executed_at": executed_at,
    }
    return sign_receipt(payload, sk, key_id)

_consumed = set()

def verify_chain(action_being_executed, approval, agent_rcpt, now: str):
    """One code path covers both receipt kinds (same envelope), then checks the
    things that only make sense TOGETHER: shared digest, freshness, consumption.
    `now` is an ISO-8601 Zulu timestamp; Zulu strings compare correctly as strings."""
    # 1. Shared envelope contract, separate authorities.
    ok, why = verify_envelope(approval, "human.approval.v1", APPROVER_KEYS)
    if not ok: return (False, f"approval: {why}")
    ok, why = verify_envelope(agent_rcpt, "agent.action.v1", AGENT_KEYS)
    if not ok: return (False, f"agent receipt: {why}")

    # 2. The join: BOTH receipts must bind the digest of the action being executed
    #    right now. A valid approval for a DIFFERENT action is substitution, and it
    #    gets its own reason — this is "the executed action changed".
    executing_digest = sha256_canonical(action_being_executed)
    if approval.get("action_digest") != executing_digest or approval.get("action") != action_being_executed:
        return (False, "digest substitution: the approval binds a different canonical action than the one being executed")
    if agent_rcpt.get("action_digest") != executing_digest or agent_rcpt.get("action") != action_being_executed:
        return (False, "digest substitution: the agent receipt binds a different canonical action than the one being executed")
    if agent_rcpt.get("parent_approval_ref") != receipt_hash(approval):
        return (False, "agent receipt is not bound to this approval")

    # 3. Freshness: a valid signature over stale authority is still a refusal —
    #    each staleness gets its own reason, distinct from substitution above.
    if approval.get("policy_version") != CURRENT_POLICY["policy_version"]:
        return (False, f"stale authority: approved under policy {approval.get('policy_version')!r}, current is {CURRENT_POLICY['policy_version']!r}")
    expires = approval.get("expires_at")
    if not isinstance(expires, str) or not expires or now >= expires:
        return (False, "stale authority: approval expired before execution")

    # 4. One-time consumption: an approval authorizes ONE execution.
    ref = receipt_hash(approval)
    if ref in _consumed:
        return (False, "approval already consumed (replay refused)")
    _consumed.add(ref)
    return (True, f"approved by {approval['approver_id']}, executed by {agent_rcpt['agent_id']}")

def execute(action, approval, agent_rcpt, now):
    ok, why = verify_chain(action, approval, agent_rcpt, now)
    return (ok, "executed" if ok else why)

receipt = agent_receipt(action, approval, "2026-07-08T15:04:06Z")
print(execute(action, approval, receipt, now="2026-07-08T15:04:07Z"))

(True, 'executed')


## Какво прихваща връзката

Всеки един от случаите по-долу се проваля **затворено** с **отделна причина**. Първият блок е класическият набор (подправяне, объркан заместник, повторно изпълнение, фалшификация на какъвто и да е орган, неправилно форматиран вход). Вторият блок е двойката, която прави свойството реално, а не само твърдено:

- **изтекъл орган** — подписът все още е валиден, но версията на политиката е преместена, ключът на одобряващия е изваден от фиксирания регистър или одобрението е изтекло преди изпълнението;
- **замяна на дайджеста** — валидно подписан получател на действие, чието `parent_approval_ref` сочи към *истинско* одобрение, но каноничният дайджест на действието за това одобрение не съвпада с действието, което действително се изпълнява.


In [6]:
NOW = "2026-07-08T15:05:00Z"

# 1. tamper: change the amount after approval — the executed action changed.
tampered = {**action, "params": {**action["params"], "amount_usd": 9900}}
print("tamper              ->", verify_chain(tampered, approval, agent_receipt(tampered, approval, NOW), NOW))

# 2. confused deputy: valid approval for action A, presented to execute action B.
action_b = {**action, "action_type": "wire.send"}
print("confused-deputy     ->", verify_chain(action_b, approval, agent_receipt(action_b, approval, NOW), NOW))

# 3. replay: the approval was consumed by the successful execution above.
print("replay              ->", execute(action, approval, agent_receipt(action, approval, NOW), NOW))

# 4. forged approval: attacker signs with their own key but claims a pinned key_id.
mallory_sk = SigningKey.generate()
forged = human_approval(action, "mallory", NOW, sk=mallory_sk)
print("forged-approval     ->", verify_chain(action, forged, agent_receipt(action, forged, NOW), NOW))

# A fresh, un-consumed approval so the agent-side cases fail on their OWN check.
fresh = human_approval(action, "alice@ops (WebAuthn)", NOW, expires_at="2026-07-08T15:20:00Z")

# 5. self-minted agent receipt: attacker's own agent key, refused by the pinned registry.
mallory_agent = agent_receipt(action, fresh, NOW, sk=SigningKey.generate())
print("self-minted-agent   ->", verify_chain(action, fresh, mallory_agent, NOW))

# 6. wrong-action agent receipt: real agent key, but the receipt binds a different action.
wrong_action = {**action, "params": {**action["params"], "amount_usd": 9900}}
print("wrong-action-agent  ->", verify_chain(action, fresh, agent_receipt(wrong_action, fresh, NOW), NOW))

# 7. malformed input: structurally broken receipts refuse cleanly, they never crash.
print("malformed-approval  ->", verify_chain(action, {"type": "human.approval.v1"}, agent_receipt(action, fresh, NOW), NOW))
print("malformed-agent     ->", verify_chain(action, fresh, {"nope": "not a receipt"}, NOW))

# 8. wrong-length signature: valid base64, not 64 bytes — refused, not crashed.
badlen = {**fresh, "signature": {**fresh["signature"], "sig": "AAAA"}}
print("wrong-len-sig       ->", verify_chain(action, badlen, agent_receipt(action, fresh, NOW), NOW))

# 9. non-object receipt: a list refuses cleanly instead of raising AttributeError.
print("nonobject-receipt   ->", verify_chain(action, [1, 2], agent_receipt(action, fresh, NOW), NOW))

print()
print("--- the two negative controls that make the property real ---")

# 10. STALE POLICY: signature still valid, but policy moved between approval and
#     execution. Authority is decided at execution time, not signing time.
CURRENT_POLICY["policy_version"] = "refunds-v4"
print("stale-policy        ->", verify_chain(action, fresh, agent_receipt(action, fresh, NOW), NOW))
CURRENT_POLICY["policy_version"] = "refunds-v3"   # restore for the cases below

# 11. STALE KEY: the approver key is rotated out of the pinned registry after
#     signing. The signature bytes still verify against the old key — but the old
#     key no longer confers authority.
rotated_out = APPROVER_KEYS.pop("approver-key-1")
print("stale-key           ->", verify_chain(action, fresh, agent_receipt(action, fresh, NOW), NOW))
APPROVER_KEYS["approver-key-1"] = rotated_out     # restore

# 12. EXPIRED: approval was valid when signed, but execution came too late.
expired = human_approval(action, "alice@ops (WebAuthn)", "2026-07-08T14:00:00Z",
                         expires_at="2026-07-08T14:01:00Z")
print("expired-approval    ->", verify_chain(action, expired, agent_receipt(action, expired, NOW), NOW))

# 13. DIGEST SUBSTITUTION: a validly signed agent receipt whose parent_approval_ref
#     points at a REAL approval — but that approval binds action B, and the agent
#     is executing action A. Distinct reason from every staleness above.
approval_b = human_approval(action_b, "alice@ops (WebAuthn)", NOW, expires_at="2026-07-08T15:20:00Z")
substituted = agent_receipt(action, approval_b, NOW)   # executing `action`, ref -> approval of action_b
print("digest-substitution ->", verify_chain(action, approval_b, substituted, NOW))

tamper              -> (False, 'digest substitution: the approval binds a different canonical action than the one being executed')
confused-deputy     -> (False, 'digest substitution: the approval binds a different canonical action than the one being executed')
replay              -> (False, 'approval already consumed (replay refused)')
forged-approval     -> (False, 'approval: signature invalid (forged, tampered, or malformed)')
self-minted-agent   -> (False, 'agent receipt: signature invalid (forged, tampered, or malformed)')
wrong-action-agent  -> (False, 'digest substitution: the agent receipt binds a different canonical action than the one being executed')
malformed-approval  -> (False, 'approval: receipt malformed (not an object with a signature object)')
malformed-agent     -> (False, 'agent receipt: receipt malformed (not an object with a signature object)')
wrong-len-sig       -> (False, 'approval: signature invalid (forged, tampered, or malformed)')
nonobject-receipt   -> (Fa

## Какво доказва това — и какво не доказва

**Доказва:** че именуван човек е одобрил *този конкретен каноничен акт* (пълен акт + дайджест, подписан с ключ, разпознат от фиксирано хранилище), и агентът е изпълнил *точно този одобрен акт* (същият дайджест, разписка обвързана с одобрението чрез `receipt_hash`, собствената конвенция на веригата на урока) — докато версията на политиката на одобрението, ключът и изтичането са все още валидни, точно веднъж. Ако някоя от страните се промени, веригата се затваря и причината за отказа ви казва **коя** характеристика е нарушена: остарял орган срещу променен акт.

**Не доказва:** че интерфейсът за одобрение е показал на човека това, което мислят, че подписват (WYSIWYS е отделен проблем), че ключът не е бил принуден или откраднат преди ротация, или че последващите ефекти съвпадат с акта. Подписан ≠ упълномощен: валиден подпис върху остаряла политика, ротиран ключ, изтекъл прозорец или различен дайджест тук не носи никаква стойност.

Двата вида разписки споделят плика на урока и един кодов път `verify_chain` нарочно: обвързването, което създадохте за разписките за действия в основния тетрадка, е същият код, който проверява човешкото одобрение. Един договор за верификация, отделни фиксирани органи, обединени от каноничния дайджест на акта и нищо друго.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Отказ от отговорност**:
Този документ е преведен с помощта на AI преводачески услуга [Co-op Translator](https://github.com/Azure/co-op-translator). Въпреки че се стремим към точност, моля имайте предвид, че автоматизираните преводи могат да съдържат грешки или неточности. Оригиналният документ на неговия роден език трябва да се счита за авторитетен източник. За критична информация се препоръчва професионален човешки превод. Ние не носим отговорност за каквито и да е недоразумения или неправилни тълкувания, произтичащи от използването на този превод.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
